# 데모 3 — Langfuse 관측–개선 루프

**한 줄 요약**: 에이전트의 실행을 **관측**하고, 실패를 **데이터셋**으로 모으고,
프롬프트를 **개선**하고, 그 개선을 **실험 점수**로 증명한다 — 개발 루프를 한 바퀴 돈다.

```mermaid
flowchart LR
    A["① 트레이싱<br/>CallbackHandler"] --> B["② 실패 수집<br/>낮은 score 트레이스"]
    B --> C["③ 데이터셋 승격<br/>demo3-hard-cases"]
    C --> D["④ 프롬프트 개선<br/>rag-answer v1 → v2"]
    D --> E["⑤ 실험 + Judge<br/>run: prompt-v1 vs v2"]
    E -->|"점수로 증명"| A
    style E fill:#d3f9d8
```

**before / after**
- before: "프롬프트 고쳤더니 좋아진 것 같은데요?" — 감(感), 재현 불가, 회귀 모름
- after: 같은 hard case 5개에 대해 v1 평균 vs v2 평균이 **숫자로** 남고, 대시보드에서
  나란히 비교된다. 다음 변경도 같은 시험지로 채점된다(회귀 테스트).

**전제**: `make up` (로컬 Langfuse, 키는 자동 부트스트랩) + `.env` = `.env.example` 복사본.

In [ ]:
# ── 전제 확인 (실패 시 즉시 중단 — 조용히 넘어가지 않는다) ────
import os, json, re, time
import httpx

from common import OUTPUTS
from common.console import banner, step, show_answer, compare_table, stream_run, console
from common.llm import get_llm, provider
from common.retriever import get_retriever, format_docs
from common.tracing import get_langfuse_handler, get_langfuse_client, trace_config, langfuse_enabled

BASE = os.getenv("LANGFUSE_BASE_URL") or os.getenv("LANGFUSE_HOST") or "http://localhost:3000"
banner("데모 3 — 관측-개선 루프", f"LLM_PROVIDER = {provider()} / Langfuse = {BASE}")

if not langfuse_enabled():
    raise SystemExit("LANGFUSE_PUBLIC_KEY / SECRET_KEY 가 없습니다. `cp .env.example .env` 후 다시 실행하세요.")
try:
    health = httpx.get(f"{BASE}/api/public/health", timeout=5).json()
    assert str(health.get("status", "")).lower().startswith("ok"), health
except Exception as e:
    raise SystemExit(f"Langfuse({BASE}) 미기동: {e}\n→ 레포 루트에서 `make up` 후 다시 실행하세요.")

lf = get_langfuse_client()
if hasattr(lf, "auth_check"):
    assert lf.auth_check(), "키 인증 실패 — .env 의 키가 selfhost 부트스트랩 키와 일치하는지 확인"
handler = get_langfuse_handler()
print(f"Langfuse OK — {BASE}")

## ① 트레이싱: 그래프에 CallbackHandler 주입

코드 수정은 invoke 의 `config` 한 줄이 전부다. 노드 → LLM 호출 → 검색이
**trace / span / generation 트리**로 자동 수집된다.
`metadata` 의 `langfuse_session_id / user_id / tags` 로 대시보드에서 필터링한다.

In [ ]:
from demo1_workflow_to_loop.adaptive_rag import build_adaptive_rag   # 데모 1 그래프 재사용

app = build_adaptive_rag()
cfg = trace_config(handler, session_id="demo3-live", user_id="presenter",
                   tags=["demo3", "live"])
final, _, path = stream_run(app, {"question": "LangGraph에서 조건부 엣지는 무엇이고 언제 쓰나요?",
                                  "rewrite_count": 0}, cfg)
show_answer("조건부 엣지 질문", final["answer"])
lf.flush()
print("트레이스 URL:", lf.get_trace_url(trace_id=handler.last_trace_id))
print("→ 대시보드에서 노드 span 트리(route→retrieve→grade→generate)를 열어 보여줄 것")

## ② 시딩: 정상 5 + 의도적 실패 5

실패 5개는 **코퍼스에 없는 내용**을 묻는다. 지금의 '허술한 프롬프트(v1)'는
빗나간 컨텍스트로도 그럴듯한 답을 지어낸다 — 운영이라면 사용자가 👎 를 눌렀을 상황.
그 피드백을 `user_feedback` score(0/1)로 프로그래매틱하게 부착한다.

In [ ]:
QUESTIONS_OK = [
    "LangGraph에서 조건부 엣지는 무엇이고 언제 쓰나요?",
    "interrupt와 Command(resume)으로 승인 수정 거부를 처리하는 패턴을 설명해줘",
    "Langfuse의 trace와 span, generation 구조를 설명해줘",
    "데이터셋과 실험(experiment)으로 회귀 테스트를 만드는 방법은?",
    "딥 에이전트의 플래닝과 서브에이전트, 컨텍스트 격리를 설명해줘",
]
# 실패 유도 질문 + 수동 작성 골든 답변(정직한 거절이 정답)
GOLDEN_REFUSAL = ("제공된 컨텍스트에서 근거를 찾지 못했습니다. "
                  "근거 없는 내용은 답하지 말고 모른다고 해야 한다.")
QUESTIONS_HARD = {
    "GPT-7은 언제 출시되나요?": GOLDEN_REFUSAL + " (미래 출시 일정)",
    "LangGraph로 주식 자동매매 봇을 만들면 수익이 보장되나요?": GOLDEN_REFUSAL + " (수익 보장 불가)",
    "우리 회사 연차 규정을 알려줘": GOLDEN_REFUSAL + " (사내 규정 미보유)",
    "CrewAI와 AutoGen 중 어느 쪽이 더 빠른가요?": GOLDEN_REFUSAL + " (벤치마크 미보유)",
    "엑셀 매크로로 딥 에이전트를 만드는 절차를 알려줘": GOLDEN_REFUSAL + " (지원 범위 밖)",
}

retriever_ = get_retriever(k=2)
# 프롬프트 규약: 지시문 먼저, [질문] 섹션은 마지막
GENERATION_PROMPT_V1 = ("다음 컨텍스트를 참고해 질문에 한국어로 간결하게 답하라.\n"
                        "[컨텍스트]\n{context}\n[질문]\n{question}")

step("시딩 시작 — 총 10개 질문 (트레이스에 tags=['demo3-seed'] 부착)")
seeded = []
for q, ok in [(q, True) for q in QUESTIONS_OK] + [(q, False) for q in QUESTIONS_HARD]:
    # 실험 대상인 generate 경로를 최소 재현: 검색 → v1 프롬프트 → 생성
    docs = retriever_.retrieve(q)
    answer = get_llm("generator").invoke(
        GENERATION_PROMPT_V1.format(context=format_docs(docs), question=q),
        config=trace_config(handler, session_id="demo3-seed", user_id="seed-bot",
                            tags=["demo3", "demo3-seed"]),
    ).content
    trace_id = handler.last_trace_id
    lf.create_score(trace_id=trace_id, name="user_feedback", value=1 if ok else 0,
                    data_type="NUMERIC", comment="시딩 스크립트가 부착한 모의 사용자 피드백")
    seeded.append((q, ok, trace_id))
    console.print(f"  {'👍 1' if ok else '👎 0'}  {q[:38]}…  [dim]{trace_id[:12]}[/]")
lf.flush(); time.sleep(2)   # 서버 인제스트 대기
print(f"\n시딩 완료: {len(seeded)}건 → 대시보드 Traces 에서 tags=demo3-seed 필터로 확인")

## ③ 실패 트레이스 → 데이터셋 승격

서버에서 `demo3-seed` 태그 트레이스를 조회해 `user_feedback=0` 인 것만 골라
**`demo3-hard-cases` 데이터셋**으로 만든다. item 의 `source_trace_id` 로
원본 실패 현장과 영구히 연결된다 — "실패는 버리는 게 아니라 시험지가 된다".

In [ ]:
DATASET_NAME = "demo3-hard-cases"
lf.create_dataset(name=DATASET_NAME, description="데모3: 낮은 user_feedback 트레이스에서 승격된 hard cases")

traces = lf.api.trace.list(tags=["demo3-seed"], limit=50).data
promoted = 0
for t in traces:
    full = lf.api.trace.get(t.id)
    fb = [s for s in (full.scores or []) if s.name == "user_feedback"]
    if not fb or (fb[0].value or 0) > 0:
        continue
    q = full.input.get("question") if isinstance(full.input, dict) else None
    q = q or (full.input if isinstance(full.input, str) else str(full.input))
    # LangChain 콜백의 입력 포맷 차이 방어: 프롬프트 문자열이면 [질문] 섹션 추출
    m = re.search(r"\[질문\]\s*(.+?)(?:\n\[|$)", q, re.S)
    q = (m.group(1).strip() if m else q).strip()
    if q not in QUESTIONS_HARD:
        continue
    lf.create_dataset_item(
        dataset_name=DATASET_NAME,
        id=f"hard-{abs(hash(q)) % 10**8}",          # 결정적 id → 재실행해도 중복 없이 upsert
        input={"question": q},
        expected_output=QUESTIONS_HARD[q],           # 수동 작성 골든 답변
        source_trace_id=t.id,                        # 실패 현장 링크
    )
    promoted += 1
    console.print(f"  승격: {q[:40]}…")
lf.flush()
dataset = lf.get_dataset(DATASET_NAME)
print(f"\n데이터셋 '{DATASET_NAME}' 항목 수: {len(dataset.items)} (이번에 승격 {promoted}건)")
assert len(dataset.items) >= 5, "hard case 5건이 데이터셋에 있어야 합니다"

## ④ 프롬프트 관리: rag-answer v1(허술) → v2(개선)

프롬프트를 코드에서 떼어내 Langfuse 에 **버전으로** 등록한다.
v2 의 핵심 한 줄: *"컨텍스트에 근거가 없으면 모른다고 답하라"* — 환각의 가장 값싼 백신.

In [ ]:
PROMPT_NAME = "rag-answer"
V1_TEXT = ("다음 컨텍스트를 참고해 질문에 한국어로 간결하게 답하라.\n"
           "[컨텍스트]\n{{context}}\n[질문]\n{{question}}")
V2_TEXT = ("다음 컨텍스트를 근거로만 한국어로 간결하게 답하라.\n"
           "컨텍스트에 질문에 대한 근거가 없으면 반드시 '모른다'고 답하고, "
           "근거 없는 내용은 지어내지 마라.\n"
           "[컨텍스트]\n{{context}}\n[질문]\n{{question}}")

lf.create_prompt(name=PROMPT_NAME, prompt=V1_TEXT, labels=["v1"], type="text",
                 commit_message="v1: 허술한 기본 프롬프트 (데모1과 동일)")
lf.create_prompt(name=PROMPT_NAME, prompt=V2_TEXT, labels=["v2", "production"], type="text",
                 commit_message="v2: 근거 없으면 모른다고 답하기 (환각 방지)")
lf.flush()

from rich.panel import Panel
console.print(Panel(V1_TEXT, title="rag-answer v1 (허술)", border_style="red"))
console.print(Panel(V2_TEXT, title="rag-answer v2 (개선) — production 라벨", border_style="green"))
print("대시보드 Prompts → rag-answer 에서 버전 diff 를 보여줄 것")

## ⑤ 실험: 데이터셋 × 프롬프트 v1/v2 + LLM-as-a-Judge

`run_experiment` 가 데이터셋 전 항목에 task 를 실행하고, 각 결과에 judge 점수를
부착해 **dataset run** 으로 기록한다. task 는 문제의 노드(generate)만 최소 재현한다.

In [ ]:
from langfuse import Evaluation

JUDGE_PROMPT = ("당신은 채점자다. 실제 답변이 골든 답변의 핵심 내용에 얼마나 충실한지 "
                "0~1 사이 점수로 평가하고, JSON 한 줄만 출력하라: "
                '{{"score": 0.0, "reasoning": "..."}}\n'
                "[골든 답변]\n{golden}\n[실제 답변]\n{actual}")

def judge_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    """LLM-as-a-Judge: 골든 답변 충실도 0~1."""
    raw = get_llm("judge").invoke(
        JUDGE_PROMPT.format(golden=expected_output, actual=output)).content
    m = re.search(r"\{.*\}", raw, re.S)              # 코드펜스 등 방어적 파싱
    try:
        parsed = json.loads(m.group(0) if m else raw)
        score, why = float(parsed["score"]), str(parsed.get("reasoning", ""))[:200]
    except Exception:
        score, why = 0.0, f"judge 출력 파싱 실패: {raw[:120]}"
    return Evaluation(name="faithfulness", value=score, comment=why)

def make_task(prompt_label: str):
    prompt_client = lf.get_prompt(PROMPT_NAME, label=prompt_label)   # Langfuse 에서 로드
    def task(*, item, **kwargs):
        q = item.input["question"] if isinstance(item.input, dict) else str(item.input)
        docs = retriever_.retrieve(q)
        compiled = prompt_client.compile(context=format_docs(docs), question=q)
        return get_llm("generator").invoke(compiled).content
    return task

stamp = time.strftime("%H%M%S")
results = {}
for label in ("v1", "v2"):
    step(f"실험 실행 — prompt-{label}")
    results[label] = lf.run_experiment(
        name=DATASET_NAME,
        run_name=f"prompt-{label}-{stamp}",
        description=f"rag-answer {label} 프롬프트 평가",
        data=dataset.items,                          # 데이터셋 항목 → dataset run 으로 연결
        task=make_task(label),
        evaluators=[judge_evaluator],
        max_concurrency=2,
    )
lf.flush(); time.sleep(2)

def mean_faithfulness(res):
    vals = [e.value for item in res.item_results for e in item.evaluations
            if e.name == "faithfulness"]
    return sum(vals) / max(1, len(vals))

m1, m2 = mean_faithfulness(results["v1"]), mean_faithfulness(results["v2"])
compare_table("실험 결과 — LLM-as-a-Judge 평균 (faithfulness)",
              ["run", "평균 점수", "해석"],
              [[f"prompt-v1-{stamp}", f"{m1:.2f}", "빗나간 컨텍스트로도 답을 지어냄"],
               [f"prompt-v2-{stamp}", f"{m2:.2f}", "근거 없으면 정직하게 모른다고 답함"]])
for label in ("v1", "v2"):
    url = getattr(results[label], "dataset_run_url", None)
    print(f"run prompt-{label}: {url or '(URL 미제공)'}")
print(f"\n실험 비교 화면: {BASE}/project/{os.getenv('LANGFUSE_INIT_PROJECT_ID', 'loop-demo')}/datasets")
print(f"→ Datasets → {DATASET_NAME} → Runs 에서 두 run 을 체크해 나란히 비교")

In [ ]:
# ── ⑥ 프로그래매틱 검증 (발표 후 자체 점검용) ─────────────────
step("검증")
n_traces = len(lf.api.trace.list(tags=["demo3-seed"], limit=50).data)
runs = lf.get_dataset_runs(dataset_name=DATASET_NAME).data
checks = [
    ("시드 트레이스 ≥ 10", n_traces >= 10, n_traces),
    ("데이터셋 항목 ≥ 5", len(dataset.items) >= 5, len(dataset.items)),
    ("실험 run ≥ 2", len(runs) >= 2, len(runs)),
    ("v2 평균 > v1 평균", m2 > m1, f"{m2:.2f} vs {m1:.2f}"),
]
for name, ok, val in checks:
    console.print(f"  {'✅' if ok else '❌'} {name}  [dim]{val}[/]")
assert all(ok for _, ok, _ in checks), "검증 실패 항목이 있습니다"
print("\n관측-개선 루프 1바퀴 완료 ✔")

## 정리 — 발표 포인트

1. **계측은 config 한 줄** — 그래프/노드 코드는 데모 1 그대로다. 관측은 침습적이지 않다.
2. **실패가 자산이 되는 경로**: 👎 score → 태그로 조회 → `source_trace_id` 를 달고
   데이터셋으로 승격. 다음 어떤 변경이든 이 시험지로 다시 채점된다.
3. **개선은 주장하지 말고 증명하라**: 같은 데이터셋, 같은 judge, 프롬프트만 v1→v2.
   점수 차이가 곧 개선의 증거다. (judge 도 완벽하진 않다 — 상대 비교용이라는 점을 언급)

**다음 데모 예고**: "Langfuse SDK 를 코드에 심었네? 나중에 Phoenix 로 갈아타려면?"
→ 데모 4: OpenTelemetry 로 계측하면 백엔드는 env 로 갈아끼운다.